# Experiment 10: input-dependent GBM/CatBoost gate

This replaces AutoGluon's single static ensemble weight with a small neural gate. For each listing, the gate receives listing context and independent base-model predictions, produces softmax weights, and minimises the blended prediction's squared residual.

The gate is trained only on chronological out-of-fold base-model predictions. It never sees in-sample base predictions, and the final chronological holdout is not used to fit either the base models or gate.

**Executed findings:** LightGBM scored **474.258**, CatBoost **470.722**, the equal blend **468.740**, and the neural gate **472.274 RMSE**. The gate learned mean weights of **0.207 LightGBM / 0.793 CatBoost** but did not beat fixed averaging. A second three-CatBoost study showed the same pattern: equal blend **471.101** versus gated **471.553**. Input-dependent gating added variance without discovering a better stable routing rule.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidate_roots if (path / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
GROUP_COLUMNS = ['TOWN', 'FLAT_TYPE']
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
compact_features = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL',
    'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month',
    'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR',
    'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]

data_dir, datasets = load_datasets(project_root / 'data')
raw_train, raw_test = datasets['train'].copy(), datasets['test'].copy()
for frame in (raw_train, raw_test):
    frame['period'] = pd.to_datetime(frame['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
test_periods = np.sort(raw_test['period'].unique())
TEST_HORIZON_MONTHS = len(test_periods)
assert TEST_HORIZON_MONTHS == 17, f'Expected 17 test months, found {TEST_HORIZON_MONTHS}'
LAG_MONTHS = 24
periods = np.sort(raw_train['period'].unique())
validation_cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw_train.loc[raw_train['period'] < validation_cutoff].copy()
raw_valid = raw_train.loc[raw_train['period'] >= validation_cutoff].copy()
print(f'Test horizon: {TEST_HORIZON_MONTHS} months; validation cutoff: {validation_cutoff}')

class TargetFeatureTransformer:
    """Leakage-safe lag and historical town/flat-type target features."""
    def __init__(self, lag_months, group_columns=GROUP_COLUMNS, target=TARGET):
        self.lag_months, self.group_columns, self.target = lag_months, group_columns, target
    def fit(self, labelled_frame):
        history = labelled_frame[self.group_columns + ['period', self.target]].copy()
        self.global_mean_ = history[self.target].mean()
        self.group_mean_ = history.groupby(self.group_columns)[self.target].mean()
        self.monthly_target_mean_ = history.groupby('period')[self.target].mean()
        self.lag_lookup_ = history.groupby(self.group_columns + ['period'], as_index=False)[self.target].mean().rename(columns={'period': 'source_period', self.target: 'town_flat_type_target_lag'})
        return self
    def transform(self, frame):
        result = frame[self.group_columns + ['period']].copy()
        result['monthly_target_lag'] = (result['period'] - self.lag_months).map(self.monthly_target_mean_).fillna(self.global_mean_)
        result['source_period'] = result['period'] - self.lag_months
        result = result.merge(self.lag_lookup_, on=self.group_columns + ['source_period'], how='left').drop(columns='source_period')
        group_mean = pd.Series(result.set_index(self.group_columns).index.map(self.group_mean_), index=result.index)
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        result['town_flat_type_target_lag'] = result['town_flat_type_target_lag'].fillna(result['town_flat_type_target_mean'])
        return result[['monthly_target_lag', 'town_flat_type_target_lag', 'town_flat_type_target_mean']]
    def fit_transform_train(self, labelled_frame):
        self.fit(labelled_frame)
        history = labelled_frame[self.group_columns + ['period', self.target]].copy()
        grouped = history.groupby(self.group_columns + ['period'], as_index=False)[self.target].agg(['sum', 'count']).reset_index().sort_values(self.group_columns + ['period'])
        grouped['prior_sum'] = grouped.groupby(self.group_columns)['sum'].cumsum() - grouped['sum']
        grouped['prior_count'] = grouped.groupby(self.group_columns)['count'].cumsum() - grouped['count']
        grouped['town_flat_type_target_mean'] = grouped['prior_sum'] / grouped['prior_count'].replace(0, np.nan)
        global_monthly = history.groupby('period')[self.target].agg(['sum', 'count']).sort_index()
        global_prior = (global_monthly['sum'].cumsum() - global_monthly['sum']) / (global_monthly['count'].cumsum() - global_monthly['count']).replace(0, np.nan)
        result = labelled_frame[self.group_columns + ['period']].copy().merge(grouped[self.group_columns + ['period', 'town_flat_type_target_mean']], on=self.group_columns + ['period'], how='left')
        result['town_flat_type_target_mean'] = result['town_flat_type_target_mean'].fillna(result['period'].map(global_prior))
        result['source_period'] = result['period'] - self.lag_months
        result = result.merge(self.lag_lookup_, on=self.group_columns + ['source_period'], how='left').drop(columns='source_period')
        result['town_flat_type_target_lag'] = result['town_flat_type_target_lag'].fillna(result['town_flat_type_target_mean'])
        result['monthly_target_lag'] = (result['period'] - self.lag_months).map(self.monthly_target_mean_).fillna(result['period'].map(global_prior))
        return result[['monthly_target_lag', 'town_flat_type_target_lag', 'town_flat_type_target_mean']]

def preprocess_partition(preprocessor, raw_frame, fit=False):
    features = raw_frame.drop(columns=['period', TARGET], errors='ignore')
    return preprocessor.fit_transform(features) if fit else preprocessor.transform(features)
def labelled_history(raw_frame, base_frame):
    result = base_frame[GROUP_COLUMNS].reset_index(drop=True).copy()
    result['period'], result[TARGET] = raw_frame['period'].to_numpy(), raw_frame[TARGET].to_numpy()
    return result
def nearest_mrt_name(base_frame):
    stations = datasets['mrt'][['NAME', 'LATITUDE', 'LONGITUDE']].dropna()
    _, indices = cKDTree(stations[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]).query(base_frame[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96])
    return pd.Series(stations['NAME'].astype(object).to_numpy()[indices], index=base_frame.index, name='nearest_mrt_name')
def nearest_geojson_name(base_frame, filename, name_field, feature_name):
    with (project_root / 'data' / 'possible_additional_features' / filename).open() as file:
        geojson = json.load(file)
    points = pd.DataFrame([{'longitude': item['geometry']['coordinates'][0], 'latitude': item['geometry']['coordinates'][1], feature_name: item['properties'][name_field]} for item in geojson['features'] if item.get('geometry', {}).get('type') == 'Point']).dropna()
    _, indices = cKDTree(points[['longitude', 'latitude']].to_numpy() * [110.96, 111.0]).query(base_frame[['LONGITUDE', 'LATITUDE']].to_numpy() * [110.96, 111.0])
    return pd.Series(points[feature_name].astype(object).to_numpy()[indices], index=base_frame.index, name=feature_name)

# Compare gates using the global 24-month lag configuration.
MODEL_FEATURES = compact_features + [
    'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name',
    'nearest_tourist_attraction_name', 'nearest_monument_name',
]
BASE_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}
BASE_TIME_LIMIT_SECONDS = int(os.environ.get('GATED_BASE_TIME_LIMIT', '600'))
N_OOF_SPLITS = int(os.environ.get('GATED_OOF_SPLITS', '3'))
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


Test horizon: 17 months; validation cutoff: 2024-05


## Leakage-safe protocol

1. Split the pre-2024-05 rows into expanding chronological folds.
2. For every fold, fit the shared feature preprocessor, target features, LightGBM, and CatBoost only on that fold's earlier rows; predict its later rows.
3. Train the gate on the earlier OOF folds and use the latest OOF fold solely for early stopping.
4. Fit the two base models on all pre-2024-05 rows, predict the untouched 2024-05+ holdout, and evaluate the gate there.

The gate predicts convex weights with softmax, so its prediction is always a weighted average of the two base predictions.

In [ ]:
def build_model_frame(base_frame, target_features):
    nearest_mrt = nearest_mrt_name(base_frame).reset_index(drop=True)
    nearest_attraction = nearest_geojson_name(
        base_frame, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name'
    ).reset_index(drop=True)
    nearest_monument = nearest_geojson_name(
        base_frame, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name'
    ).reset_index(drop=True)
    return pd.concat([
        base_frame[compact_features].reset_index(drop=True), target_features.reset_index(drop=True),
        nearest_mrt, nearest_attraction, nearest_monument,
    ], axis=1)[MODEL_FEATURES]

def one_hot_encode_town(train_frame, later_frame):
    """Fit town one-hot columns on an earlier partition and transform a later one."""
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.int8)
    train_town = train_frame[['TOWN']].fillna('__missing_town__')
    later_town = later_frame[['TOWN']].fillna('__missing_town__')
    train_encoded = pd.DataFrame(
        encoder.fit_transform(train_town), index=train_frame.index,
        columns=[f'town_onehot__{name}' for name in encoder.get_feature_names_out(['TOWN'])],
    )
    later_encoded = pd.DataFrame(
        encoder.transform(later_town), index=later_frame.index, columns=train_encoded.columns,
    )
    # Base-model functions exclude raw TOWN; retain it here only as context for the neural gate.
    return (pd.concat([train_frame, train_encoded], axis=1),
            pd.concat([later_frame, later_encoded], axis=1))

def make_fold_frames(earlier_rows, later_rows):
    fold_preprocessor = RentalPreprocessor(datasets)
    earlier_base = preprocess_partition(fold_preprocessor, earlier_rows, fit=True)
    later_base = preprocess_partition(fold_preprocessor, later_rows)
    feature_transformer = TargetFeatureTransformer(LAG_MONTHS)
    earlier_features = feature_transformer.fit_transform_train(labelled_history(earlier_rows, earlier_base))
    later_features = feature_transformer.transform(labelled_history(later_rows, later_base))
    return one_hot_encode_town(
        build_model_frame(earlier_base, earlier_features),
        build_model_frame(later_base, later_features),
    )

def fit_base_models(train_x, train_y, model_name):
    predictor = TabularPredictor(
        label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_10_gated_models' / RUN_ID / model_name),
    ).fit(
        train_data=train_x.drop(columns='TOWN').assign(**{TARGET: train_y.to_numpy()}),
        hyperparameters=BASE_HYPERPARAMETERS, fit_weighted_ensemble=False,
        presets=['medium_quality', 'ignore_text'], time_limit=BASE_TIME_LIMIT_SECONDS, verbosity=2,
    )
    names = predictor.model_names()
    gbm_name = next(name for name in names if name.startswith('LightGBM'))
    cat_name = next(name for name in names if name.startswith('CatBoost'))
    return predictor, gbm_name, cat_name

def base_predictions(predictor, gbm_name, cat_name, frame):
    return pd.DataFrame({
        'gbm_prediction': predictor.predict(frame.drop(columns='TOWN'), model=gbm_name).astype(float).to_numpy(),
        'cat_prediction': predictor.predict(frame.drop(columns='TOWN'), model=cat_name).astype(float).to_numpy(),
    })


In [11]:
# OOF rows are drawn only from raw_fit, which ends before the untouched final holdout.
fit_periods = np.array(sorted(raw_fit['period'].unique()))
oof_parts = []
for fold, (train_index, eval_index) in enumerate(TimeSeriesSplit(n_splits=N_OOF_SPLITS).split(fit_periods), start=1):
    fold_train = raw_fit[raw_fit['period'].isin(fit_periods[train_index])].copy()
    fold_eval = raw_fit[raw_fit['period'].isin(fit_periods[eval_index])].copy()
    fold_train_x, fold_eval_x = make_fold_frames(fold_train, fold_eval)
    predictor, gbm_name, cat_name = fit_base_models(
        fold_train_x, fold_train[TARGET].astype(float).reset_index(drop=True), f'oof_fold_{fold}'
    )
    fold_base_predictions = base_predictions(predictor, gbm_name, cat_name, fold_eval_x)
    fold_context = fold_eval_x.reset_index(drop=True).copy()
    fold_context['target'] = fold_eval[TARGET].to_numpy()
    fold_context['fold'] = fold
    oof_parts.append(pd.concat([fold_context, fold_base_predictions], axis=1))
    print(f'OOF fold {fold}: train through {fit_periods[train_index[-1]]}; evaluate {len(fold_eval):,} rows')

oof = pd.concat(oof_parts, ignore_index=True)
assert oof[['gbm_prediction', 'cat_prediction']].notna().all().all()
display(oof[['fold', 'target', 'gbm_prediction', 'cat_prediction']].head())


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.95 GB / 16.00 GB (30.9%)
Disk Space Avail:   181.52 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_gated_models/20260927_110706/oof_fold_1"
Train Data Rows:    31565
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    5092.82 MB
	Train Data (Original)  Memory Usage: 17.34 MB (0.3% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 26 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMin

OOF fold 1: train through 2021-10; evaluate 28,514 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.92 GB / 16.00 GB (30.7%)
Disk Space Avail:   181.50 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_gated_models/20260927_110706/oof_fold_2"
Train Data Rows:    60079
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to p

OOF fold 2: train through 2022-08; evaluate 27,858 rows
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.82 GB / 16.00 GB (30.1%)
Disk Space Avail:   181.49 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_gated_models/20260927_110706/oof_fold_3"
Train Data Rows:    87937
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to p

OOF fold 3: train through 2023-06; evaluate 30,051 rows


,fold,target,gbm_prediction,cat_prediction
0,1,2000,2224.556641,2197.537109
1,1,2600,2613.929199,2657.000732
2,1,2100,2156.296875,2088.433838
3,1,2550,2369.257568,2498.737305
4,1,2000,2408.541504,2232.899414


In [12]:
# Town is deliberately one-hot encoded for the neural gate.
GATE_TOWN_COLUMNS = ['TOWN']
GATE_CATEGORICAL_COLUMNS = [
    'FLAT_TYPE', 'FLAT_MODEL', 'nearest_mrt_name',
    'nearest_tourist_attraction_name', 'nearest_monument_name',
]
GATE_NUMERIC_COLUMNS = [
    'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'YEAR_COMPLETED', 'MAX_FLOOR',
    'coe_quota_premium_mean', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
    'monthly_target_lag', 'town_flat_type_target_mean', 'gbm_prediction', 'cat_prediction',
]
GATE_INPUT_COLUMNS = GATE_NUMERIC_COLUMNS + GATE_TOWN_COLUMNS + GATE_CATEGORICAL_COLUMNS

def make_gate_encoder():
    return ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), GATE_NUMERIC_COLUMNS),
        ('town_onehot', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), GATE_TOWN_COLUMNS),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), GATE_CATEGORICAL_COLUMNS),
    ], sparse_threshold=0)

class SoftmaxGate(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, 2),
        )

    def forward(self, context, base_predictions):
        weights = torch.softmax(self.network(context), dim=1)
        blended = (weights * base_predictions).sum(dim=1)
        return blended, weights

def train_gate(train_context, train_base, train_target, stop_context, stop_base, stop_target):
    gate = SoftmaxGate(train_context.shape[1])
    optimiser = torch.optim.AdamW(gate.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(train_context, train_base, train_target), batch_size=1024, shuffle=True)
    best_state, best_loss, stale_epochs = None, np.inf, 0
    for epoch in range(300):
        gate.train()
        for context, base, target in loader:
            prediction, _ = gate(context, base)
            loss = ((prediction - target) ** 2).mean()
            optimiser.zero_grad(); loss.backward(); optimiser.step()
        gate.eval()
        with torch.no_grad():
            stop_prediction, _ = gate(stop_context, stop_base)
            stop_loss = ((stop_prediction - stop_target) ** 2).mean().item()
        if stop_loss < best_loss - 1e-3:
            best_loss, best_state, stale_epochs = stop_loss, {k: v.detach().clone() for k, v in gate.state_dict().items()}, 0
        else:
            stale_epochs += 1
            if stale_epochs >= 30:
                break
    gate.load_state_dict(best_state)
    return gate, epoch + 1


In [13]:
# The latest OOF fold is gate-only early stopping data; it is not used for fitting gate weights.
gate_stop_fold = oof['fold'].max()
gate_train_frame = oof[oof['fold'] < gate_stop_fold].copy()
gate_stop_frame = oof[oof['fold'] == gate_stop_fold].copy()
gate_encoder = make_gate_encoder().fit(gate_train_frame[GATE_INPUT_COLUMNS])
to_tensor = lambda array: torch.tensor(np.asarray(array), dtype=torch.float32)
gate, trained_epochs = train_gate(
    to_tensor(gate_encoder.transform(gate_train_frame[GATE_INPUT_COLUMNS])),
    to_tensor(gate_train_frame[['gbm_prediction', 'cat_prediction']]), to_tensor(gate_train_frame['target']),
    to_tensor(gate_encoder.transform(gate_stop_frame[GATE_INPUT_COLUMNS])),
    to_tensor(gate_stop_frame[['gbm_prediction', 'cat_prediction']]), to_tensor(gate_stop_frame['target']),
)

# Independent final chronological holdout: base models see raw_fit only.
final_train_x, final_valid_x = make_fold_frames(raw_fit, raw_valid)
final_predictor, final_gbm_name, final_cat_name = fit_base_models(
    final_train_x, raw_fit[TARGET].astype(float).reset_index(drop=True), 'final_holdout_base_models'
)
holdout_base = base_predictions(final_predictor, final_gbm_name, final_cat_name, final_valid_x)
holdout_context = final_valid_x.reset_index(drop=True).copy()
holdout_context[['gbm_prediction', 'cat_prediction']] = holdout_base
with torch.no_grad():
    gated_prediction, gated_weights = gate(
        to_tensor(gate_encoder.transform(holdout_context[GATE_INPUT_COLUMNS])),
        to_tensor(holdout_base[['gbm_prediction', 'cat_prediction']]),
    )
holdout_context['gated_prediction'] = gated_prediction.numpy()
holdout_context['gbm_weight'] = gated_weights[:, 0].numpy()
holdout_context['cat_weight'] = gated_weights[:, 1].numpy()
actual = raw_valid[TARGET].astype(float).reset_index(drop=True)
def metric_row(name, prediction):
    return {'model': name, 'RMSE': mean_squared_error(actual, prediction) ** 0.5, 'MAE': mean_absolute_error(actual, prediction), 'R2': r2_score(actual, prediction)}
comparison = pd.DataFrame([
    metric_row('LightGBM', holdout_base['gbm_prediction']),
    metric_row('CatBoost', holdout_base['cat_prediction']),
    metric_row('Equal 50/50 blend', holdout_base[['gbm_prediction', 'cat_prediction']].mean(axis=1)),
    metric_row('Neural softmax gate', holdout_context['gated_prediction']),
])
display(comparison.sort_values('RMSE'))
display(holdout_context[['TOWN', 'FLAT_TYPE', 'gbm_weight', 'cat_weight']].groupby(['TOWN', 'FLAT_TYPE']).mean().sort_values('cat_weight', ascending=False).head(15))
print(f'Gate trained for {trained_epochs} epochs; mean holdout weights: GBM={holdout_context.gbm_weight.mean():.3f}, CatBoost={holdout_context.cat_weight.mean():.3f}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.96 GB / 16.00 GB (31.0%)
Disk Space Avail:   181.46 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_gated_models/20260927_110706/final_holdout_base_models"
Train Data Rows:    117988
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature

,model,RMSE,MAE,R2
2,Equal 50/50 blend,468.739537,341.413355,0.490971
1,CatBoost,470.722283,343.932084,0.486655
3,Neural softmax gate,472.273714,341.838519,0.483266
0,LightGBM,474.258013,345.890043,0.478915


,,gbm_weight,cat_weight
TOWN,FLAT_TYPE,,
bishan,executive,7.114793e-15,1.000000
geylang,executive,9.742774e-11,1.000000
bukit timah,executive,2.916195e-16,1.000000
central,5 room,3.061543e-16,1.000000
kallang whampoa,executive,1.338082e-08,1.000000
serangoon,2 room,6.207621e-09,1.000000
toa payoh,executive,4.366721e-09,1.000000
queenstown,executive,2.412451e-27,1.000000
bukit merah,1 room,9.735481e-07,0.999999


Gate trained for 89 epochs; mean holdout weights: GBM=0.207, CatBoost=0.793


## Gate several diverse CatBoost models

This section uses three CatBoost models with deliberately different training data and feature views. Their predictions remain out-of-fold for gate training, so the gate is not rewarded for fitting in-sample base predictions.

In [14]:
# Variants differ by training sample, random seed/depth, and available columns.
CAT_VARIANTS = {
    'cat_all_features': {'feature_view': 'all', 'sample_fraction': 1.00, 'seed': 11, 'params': {'iterations': 900, 'depth': 7, 'learning_rate': 0.05}},
    'cat_sampled_rows': {'feature_view': 'all', 'sample_fraction': 0.70, 'seed': 29, 'params': {'iterations': 1100, 'depth': 9, 'learning_rate': 0.04}},
    'cat_compact_view': {'feature_view': 'compact', 'sample_fraction': 1.00, 'seed': 53, 'params': {'iterations': 750, 'depth': 6, 'learning_rate': 0.06}},
}
CAT_PREDICTION_COLUMNS = list(CAT_VARIANTS)

def variant_feature_columns(frame, variant):
    town_columns = [column for column in frame.columns if column.startswith('town_onehot__')]
    if variant['feature_view'] == 'all':
        return [column for column in frame.columns if column != 'TOWN']
    compact_view = [column for column in compact_features if column != 'TOWN'] + [
        'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name'
    ]
    return compact_view + town_columns

def fit_cat_variant(train_x, train_y, variant_name, variant, run_name):
    feature_columns = variant_feature_columns(train_x, variant)
    selected_rows = train_x.sample(frac=variant['sample_fraction'], random_state=variant['seed']).index
    selected_x = train_x.loc[selected_rows, feature_columns]
    selected_y = train_y.loc[selected_rows]
    predictor = TabularPredictor(
        label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_10_diverse_cat_models' / RUN_ID / run_name / variant_name),
    ).fit(
        train_data=selected_x.assign(**{TARGET: selected_y.to_numpy()}),
        hyperparameters={'CAT': {**variant['params'], 'random_seed': variant['seed']}},
        fit_weighted_ensemble=False, presets=['medium_quality', 'ignore_text'],
        time_limit=BASE_TIME_LIMIT_SECONDS, verbosity=2,
    )
    model_name = next(name for name in predictor.model_names() if name.startswith('CatBoost'))
    return predictor, model_name, feature_columns

# Generate chronological OOF predictions for every CatBoost variant.
diverse_oof_parts = []
for fold, (train_index, eval_index) in enumerate(TimeSeriesSplit(n_splits=N_OOF_SPLITS).split(fit_periods), start=1):
    fold_train = raw_fit[raw_fit['period'].isin(fit_periods[train_index])].copy()
    fold_eval = raw_fit[raw_fit['period'].isin(fit_periods[eval_index])].copy()
    fold_train_x, fold_eval_x = make_fold_frames(fold_train, fold_eval)
    fold_predictions = {}
    for variant_name, variant in CAT_VARIANTS.items():
        predictor, model_name, feature_columns = fit_cat_variant(
            fold_train_x, fold_train[TARGET].astype(float).reset_index(drop=True), variant_name, variant, f'oof_fold_{fold}'
        )
        fold_predictions[variant_name] = predictor.predict(fold_eval_x[feature_columns], model=model_name).astype(float).to_numpy()
    context = fold_eval_x.reset_index(drop=True).copy()
    context['target'], context['fold'] = fold_eval[TARGET].to_numpy(), fold
    diverse_oof_parts.append(pd.concat([context, pd.DataFrame(fold_predictions)], axis=1))
    print(f'Diverse-Cat OOF fold {fold} completed.')
diverse_oof = pd.concat(diverse_oof_parts, ignore_index=True)
assert diverse_oof[CAT_PREDICTION_COLUMNS].notna().all().all()

DIVERSE_GATE_NUMERIC_COLUMNS = GATE_NUMERIC_COLUMNS[:-2] + CAT_PREDICTION_COLUMNS
DIVERSE_GATE_INPUT_COLUMNS = DIVERSE_GATE_NUMERIC_COLUMNS + GATE_TOWN_COLUMNS + GATE_CATEGORICAL_COLUMNS
def make_diverse_gate_encoder():
    return ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), DIVERSE_GATE_NUMERIC_COLUMNS),
        ('town_onehot', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), GATE_TOWN_COLUMNS),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), GATE_CATEGORICAL_COLUMNS),
    ], sparse_threshold=0)
class MultiModelSoftmaxGate(nn.Module):
    def __init__(self, input_size, num_models, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(input_size, hidden_size), nn.ReLU(), nn.Dropout(0.10), nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, num_models))
    def forward(self, context, base_predictions):
        weights = torch.softmax(self.network(context), dim=1)
        return (weights * base_predictions).sum(dim=1), weights
def train_multi_gate(train_context, train_base, train_target, stop_context, stop_base, stop_target):
    gate = MultiModelSoftmaxGate(train_context.shape[1], train_base.shape[1])
    optimiser = torch.optim.AdamW(gate.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(train_context, train_base, train_target), batch_size=1024, shuffle=True)
    best_state, best_loss, stale = None, np.inf, 0
    for epoch in range(300):
        gate.train()
        for context, base, target in loader:
            prediction, _ = gate(context, base); loss = ((prediction - target) ** 2).mean()
            optimiser.zero_grad(); loss.backward(); optimiser.step()
        gate.eval()
        with torch.no_grad():
            stop_prediction, _ = gate(stop_context, stop_base); stop_loss = ((stop_prediction - stop_target) ** 2).mean().item()
        if stop_loss < best_loss - 1e-3:
            best_loss, best_state, stale = stop_loss, {k: v.detach().clone() for k, v in gate.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= 30: break
    gate.load_state_dict(best_state)
    return gate, epoch + 1

# The final OOF fold is gate early stopping only.
stop_fold = diverse_oof['fold'].max()
diverse_train, diverse_stop = diverse_oof.query('fold < @stop_fold'), diverse_oof.query('fold == @stop_fold')
diverse_encoder = make_diverse_gate_encoder().fit(diverse_train[DIVERSE_GATE_INPUT_COLUMNS])
as_tensor = lambda values: torch.tensor(np.asarray(values), dtype=torch.float32)
diverse_gate, diverse_epochs = train_multi_gate(
    as_tensor(diverse_encoder.transform(diverse_train[DIVERSE_GATE_INPUT_COLUMNS])), as_tensor(diverse_train[CAT_PREDICTION_COLUMNS]), as_tensor(diverse_train['target']),
    as_tensor(diverse_encoder.transform(diverse_stop[DIVERSE_GATE_INPUT_COLUMNS])), as_tensor(diverse_stop[CAT_PREDICTION_COLUMNS]), as_tensor(diverse_stop['target']),
)

# Refit the variants on raw_fit and evaluate once on the untouched chronological holdout.
final_train_x, final_valid_x = make_fold_frames(raw_fit, raw_valid)
holdout_predictions = {}
for variant_name, variant in CAT_VARIANTS.items():
    predictor, model_name, feature_columns = fit_cat_variant(final_train_x, raw_fit[TARGET].astype(float).reset_index(drop=True), variant_name, variant, 'final_holdout')
    holdout_predictions[variant_name] = predictor.predict(final_valid_x[feature_columns], model=model_name).astype(float).to_numpy()
diverse_holdout = final_valid_x.reset_index(drop=True).copy()
diverse_holdout = pd.concat([diverse_holdout, pd.DataFrame(holdout_predictions)], axis=1)
with torch.no_grad():
    diverse_prediction, diverse_weights = diverse_gate(as_tensor(diverse_encoder.transform(diverse_holdout[DIVERSE_GATE_INPUT_COLUMNS])), as_tensor(diverse_holdout[CAT_PREDICTION_COLUMNS]))
diverse_holdout['gated_prediction'] = diverse_prediction.numpy()
for i, column in enumerate(CAT_PREDICTION_COLUMNS): diverse_holdout[f'{column}_weight'] = diverse_weights[:, i].numpy()
actual = raw_valid[TARGET].astype(float).reset_index(drop=True)
diverse_comparison = pd.DataFrame([
    metric_row(column, diverse_holdout[column]) for column in CAT_PREDICTION_COLUMNS
] + [metric_row('Equal three-Cat blend', diverse_holdout[CAT_PREDICTION_COLUMNS].mean(axis=1)), metric_row('Diverse-Cat neural gate', diverse_holdout['gated_prediction'])])
display(diverse_comparison.sort_values('RMSE'))
display(diverse_holdout[[f'{name}_weight' for name in CAT_PREDICTION_COLUMNS]].describe().T)
print(f'Diverse-Cat gate trained for {diverse_epochs} epochs.')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.94 GB / 16.00 GB (30.9%)
Disk Space Avail:   181.43 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_diverse_cat_models/20260927_110706/oof_fold_1/cat_all_features"
Train Data Rows:    31565
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using 

Diverse-Cat OOF fold 1 completed.
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/unders

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.72 GB / 16.00 GB (29.5%)
Disk Space Avail:   181.39 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_diverse_cat_models/20260927_110706/oof_fold_2/cat_all_features"
Train Data Rows:    60079
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using 

Diverse-Cat OOF fold 2 completed.
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/unders

...

Diverse-Cat OOF fold 3 completed.
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/unders

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.91 GB / 16.00 GB (30.7%)
Disk Space Avail:   181.29 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_10_diverse_cat_models/20260927_110706/final_holdout/cat_all_features"
Train Data Rows:    117988
Train Data Columns: 43
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Us

,model,RMSE,MAE,R2
3,Equal three-Cat blend,471.101473,344.382935,0.485828
0,cat_all_features,471.375100,344.789831,0.485230
4,Diverse-Cat neural gate,471.553114,344.694620,0.484842
2,cat_compact_view,472.278733,345.007372,0.483255
1,cat_sampled_rows,472.752646,346.190230,0.482217


,count,mean,std,min,25%,50%,75%,max
cat_all_features_weight,32012.0,0.239774,0.116079,0.000304,0.158347,0.271379,0.337655,0.425191
cat_sampled_rows_weight,32012.0,0.183710,0.111430,0.000039,0.091526,0.188005,0.282332,0.389492
cat_compact_view_weight,32012.0,0.576516,0.224868,0.222334,0.380164,0.539320,0.748580,0.999648


Diverse-Cat gate trained for 33 epochs.
